# Lezione 2 · La piccola rete

Tre cerchi sul foglio: lo slancio, la pioggia che pesa e la decisione. Poi la piega, la scotta della temperatura, la soglia e la calibrazione.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## Uno strato in forma di matrice

*Nel racconto: due cerchi che guardano la stessa serata, ciascuno con i suoi pesi.*

Lo slancio nella prima riga, la pioggia che pesa nella seconda.

In [2]:
W1 = np.array([[2.0, 0.0, 1.0],     # lo slancio: energia, pioggia, compagno
               [-3.0, 2.0, 0.0]])   # la pioggia che pesa
b1 = np.array([-1.0, 0.5])
w2 = np.array([1.5, -2.0])          # la decisione
b2 = -0.5
relu = lambda v: np.maximum(0, v)

serata = np.array([0.8, 1, 1])
h = relu(W1 @ serata + b1)
controlla("lo slancio", h[0], 1.6)
controlla("la pioggia che pesa", h[1], 0.1)
print("regola delle dimensioni:", W1.shape, "per", serata.reshape(3, 1).shape, "dà", (W1 @ serata.reshape(3, 1)).shape)

# cento serate insieme: una sola moltiplicazione tra matrici
X = np.random.default_rng(0).random((100, 3))
H = relu(X @ W1.T + b1)
assert H.shape == (100, 2)
print("cento serate per tre fatti:", X.shape, "→", H.shape)

✓  lo slancio: calcolato 1.6, nel libro 1.6
✓  la pioggia che pesa: calcolato 0.1, nel libro 0.1
regola delle dimensioni: (2, 3) per (3, 1) dà (2, 1)
cento serate per tre fatti: (100, 3) → (100, 2)


## La piccola rete in una riga

*Nel racconto: i tre cerchi, e la serata che li attraversa.*

In [3]:
def rete(x, W1=W1, b1=b1, w2=w2, b2=b2):
    h = relu(W1 @ np.array(x, dtype=float) + b1)
    z = float(w2 @ h) + b2
    return h, z, sigmoide(z)

h, z, y = rete(serata)
controlla("la spinta della decisione", z, 1.7)
controlla("e^−1,7", math.exp(-1.7), 0.183, 0.001)
controlla("la vela", y, 0.845, 0.001)

# le quattro serate della tabella del racconto
tabella = [("in forma, piove, compagno",   (0.8, 1, 1), 1.6, 0.1,  1.7, 85),
           ("in forma, asciutto, compagno", (0.8, 0, 1), 1.6, 0.0,  1.9, 87),
           ("stanco, piove, compagno",      (0.3, 1, 1), 0.6, 1.6, -2.8,  6),
           ("stanco, asciutto, compagno",   (0.3, 0, 1), 0.6, 0.0,  0.4, 60)]
for nome, x, slancio, pioggia_che_pesa, spinta, vado in tabella:
    h, z, y = rete(x)
    controlla(f"{nome}: slancio", h[0], slancio)
    controlla(f"{nome}: pioggia che pesa", h[1], pioggia_che_pesa)
    controlla(f"{nome}: spinta", z, spinta)
    controlla(f"{nome}: vado? (%)", y * 100, vado, 0.5)

✓  la spinta della decisione: calcolato 1.7, nel libro 1.7
✓  e^−1,7: calcolato 0.1827, nel libro 0.183
✓  la vela: calcolato 0.8455, nel libro 0.845
✓  in forma, piove, compagno: slancio: calcolato 1.6, nel libro 1.6
✓  in forma, piove, compagno: pioggia che pesa: calcolato 0.1, nel libro 0.1
✓  in forma, piove, compagno: spinta: calcolato 1.7, nel libro 1.7
✓  in forma, piove, compagno: vado? (%): calcolato 84.55, nel libro 85
✓  in forma, asciutto, compagno: slancio: calcolato 1.6, nel libro 1.6
✓  in forma, asciutto, compagno: pioggia che pesa: calcolato 0, nel libro 0
✓  in forma, asciutto, compagno: spinta: calcolato 1.9, nel libro 1.9
✓  in forma, asciutto, compagno: vado? (%): calcolato 86.99, nel libro 87
✓  stanco, piove, compagno: slancio: calcolato 0.6, nel libro 0.6
✓  stanco, piove, compagno: pioggia che pesa: calcolato 1.6, nel libro 1.6
✓  stanco, piove, compagno: spinta: calcolato -2.8, nel libro -2.8
✓  stanco, piove, compagno: vado? (%): calcolato 5.732, nel libro 6


## L'interazione

*Nel racconto: la pioggia che costa due punti a chi è in forma e cinquantaquattro a chi è stanco.*

In [4]:
vado = lambda x: round(rete(x)[2] * 100)
conta("punti persi per la pioggia, in forma", vado((0.8, 0, 1)) - vado((0.8, 1, 1)), 2)
conta("punti persi per la pioggia, stanco", vado((0.3, 0, 1)) - vado((0.3, 1, 1)), 54)

dz = lambda e: rete((e, 1, 1))[1] - rete((e, 0, 1))[1]
controlla("effetto della pioggia sulla spinta, in forma", dz(0.8), -0.2)
controlla("effetto della pioggia sulla spinta, stanco", dz(0.3), -3.2)

# la regressione logistica con il prodotto e·p: due equazioni, due incognite
w_ep = (dz(0.8) - dz(0.3)) / (0.8 - 0.3)
w_p = dz(0.8) - 0.8 * w_ep
controlla("w_ep", w_ep, 6)
controlla("w_p", w_p, -5.0)
controlla("con la pioggia, la pioggia che pesa è accesa per e <", (2 + 0.5) / 3, 0.83)
controlla("senza pioggia, per e <", 0.5 / 3, 0.17)

✓  punti persi per la pioggia, in forma: calcolato 2, nel libro 2
✓  punti persi per la pioggia, stanco: calcolato 54, nel libro 54
✓  effetto della pioggia sulla spinta, in forma: calcolato -0.2, nel libro -0.2
✓  effetto della pioggia sulla spinta, stanco: calcolato -3.2, nel libro -3.2
✓  w_ep: calcolato 6, nel libro 6
✓  w_p: calcolato -5, nel libro -5
✓  con la pioggia, la pioggia che pesa è accesa per e <: calcolato 0.8333, nel libro 0.83
✓  senza pioggia, per e <: calcolato 0.1667, nel libro 0.17


## Contare i parametri

*Nel racconto: undici numeri; i segni che si moltiplicano per quattro.*

In [5]:
def parametri(larghezze):
    return sum(a * c + c for a, c in zip(larghezze, larghezze[1:]))
def segni(larghezze):
    return sum(a * c for a, c in zip(larghezze, larghezze[1:]))

conta("3 → 2 → 1: segni", segni([3, 2, 1]), 8)
conta("3 → 2 → 1: parametri", parametri([3, 2, 1]), 11)
conta("3 → 4 → 3: segni", segni([3, 4, 3]), 24)
conta("3 → 4 → 3: parametri", parametri([3, 4, 3]), 31)
conta("tra due strati da 4", 4 * 4, 16)
conta("tra due strati da 8", 8 * 8, 64)
controlla("tra due strati da 4.096 (milioni)", 4096 ** 2 / 1e6, 16.8, 0.05)

✓  3 → 2 → 1: segni: calcolato 8, nel libro 8
✓  3 → 2 → 1: parametri: calcolato 11, nel libro 11
✓  3 → 4 → 3: segni: calcolato 24, nel libro 24
✓  3 → 4 → 3: parametri: calcolato 31, nel libro 31
✓  tra due strati da 4: calcolato 16, nel libro 16
✓  tra due strati da 8: calcolato 64, nel libro 64
✓  tra due strati da 4.096 (milioni): calcolato 16.78, nel libro 16.8


## I pesi iniziali

*Nel racconto: i due allievi di corporatura diversa; «a caso, ma con misura».*

A ogni strato il segnale si moltiplica per r = n · Var(w) / 2.

In [6]:
controlla("He, n = 3: deviazione dei pesi", math.sqrt(2 / 3), 0.82)
controlla("He, n = 4.096: deviazione dei pesi", math.sqrt(2 / 4096), 0.022, 0.0005)

r = lambda n, var: n * var / 2
controlla("He, n = 3: r", r(3, 0.667), 1)
controlla("deviazione 1, n = 3: r", r(3, 1), 1.5)
controlla("   dopo 20 strati", 1.5 ** 20, 3300, 50)
controlla("il doppio di He: r", r(10, 4 / 10), 2)
controlla("   dopo 20 strati (milioni)", 2 ** 20 / 1e6, 1, 0.05)
controlla("deviazione 1, n = 4.096: r", r(4096, 1), 2048)
controlla("deviazione 0,01, n = 4.096: r", r(4096, 0.01 ** 2), 0.2)
controlla("   dopo 10 strati (decimilionesimi)", 0.2 ** 10 * 1e7, 1, 0.05)

# la verifica al banco: 20 strati larghi 256, con la ReLU, pesi di He
rng = np.random.default_rng(0)
x = rng.standard_normal(256)
partenza = np.mean(x ** 2)
for _ in range(20):
    x = np.maximum(0, rng.standard_normal((256, 256)) * math.sqrt(2 / 256) @ x)
print(f"   con He, dopo 20 strati il segnale è {np.mean(x ** 2) / partenza:.2f} volte quello di partenza: né esploso né spento")

controlla("Glorot, strato 3 → 2: varianza", 2 / (3 + 2), 0.4)
controlla("Glorot, strato 3 → 2: deviazione", math.sqrt(0.4), 0.63)
controlla("estrazione uniforme tra −a e a, n = 3: a", math.sqrt(6 / 3), 1.41)

✓  He, n = 3: deviazione dei pesi: calcolato 0.8165, nel libro 0.82
✓  He, n = 4.096: deviazione dei pesi: calcolato 0.0221, nel libro 0.022
✓  He, n = 3: r: calcolato 1.001, nel libro 1
✓  deviazione 1, n = 3: r: calcolato 1.5, nel libro 1.5
✓     dopo 20 strati: calcolato 3325, nel libro 3300
✓  il doppio di He: r: calcolato 2, nel libro 2
✓     dopo 20 strati (milioni): calcolato 1.049, nel libro 1
✓  deviazione 1, n = 4.096: r: calcolato 2048, nel libro 2048
✓  deviazione 0,01, n = 4.096: r: calcolato 0.2048, nel libro 0.2
✓     dopo 10 strati (decimilionesimi): calcolato 1.024, nel libro 1
   con He, dopo 20 strati il segnale è 1.06 volte quello di partenza: né esploso né spento
✓  Glorot, strato 3 → 2: varianza: calcolato 0.4, nel libro 0.4
✓  Glorot, strato 3 → 2: deviazione: calcolato 0.6325, nel libro 0.63
✓  estrazione uniforme tra −a e a, n = 3: a: calcolato 1.414, nel libro 1.41


## La potatura

*Nel racconto: «si pota solo dopo aver visto chi è cresciuto».*

In [7]:
tutti = np.concatenate([W1.ravel(), b1, w2, [b2]])
conta("parametri", tutti.size, 11)
conta("parametri dopo aver tolto i due pesi a zero", tutti.size - int(np.sum(W1 == 0)), 9)

✓  parametri: calcolato 11, nel libro 11
✓  parametri dopo aver tolto i due pesi a zero: calcolato 9, nel libro 9


## Perché un neurone solo non forma le coppie

*Nel racconto: le coppie per il kumite, e il manico del pennello che non trova la linea.*

Il libro lo dimostra; qui si prova a tappeto: nessuna terna di pesi su una griglia fitta dà le quattro risposte giuste.

In [8]:
coppie = {(0, 0): 1, (1, 1): 1, (0, 1): 0, (1, 0): 0}
griglia = np.arange(-5, 5.01, 0.25)
trovati = sum(all((w1 * A + w2_ * B + b > 0) == bool(utile) for (A, B), utile in coppie.items())
              for w1 in griglia for w2_ in griglia for b in griglia)
conta(f"neuroni che formano le coppie, su {griglia.size ** 3} provati", trovati, 0)

✓  neuroni che formano le coppie, su 68921 provati: calcolato 0, nel libro 0


## La piega

*Nel racconto: il foglio piegato lungo la diagonale.*

In [9]:
def utile(A, B):
    h1, h2 = max(0, A - B), max(0, B - A)
    assert h1 + h2 == abs(A - B)
    return h1, h2, sigmoide(5 - 10 * (h1 + h2))

for (A, B), atteso in [((0, 0), 0.993), ((1, 1), 0.993), ((1, 0), 0.007), ((0, 1), 0.007)]:
    h1, h2, y = utile(A, B)
    controlla(f"coppia ({A}, {B}): h₁ = {h1}, h₂ = {h2}, utile?", y, atteso, 0.001)
controlla("livelli intermedi, A = 0,5 e B = 0,7", utile(0.5, 0.7)[2], 0.95)

✓  coppia (0, 0): h₁ = 0, h₂ = 0, utile?: calcolato 0.9933, nel libro 0.993
✓  coppia (1, 1): h₁ = 0, h₂ = 0, utile?: calcolato 0.9933, nel libro 0.993
✓  coppia (1, 0): h₁ = 1, h₂ = 0, utile?: calcolato 0.006693, nel libro 0.007
✓  coppia (0, 1): h₁ = 0, h₂ = 1, utile?: calcolato 0.006693, nel libro 0.007
✓  livelli intermedi, A = 0,5 e B = 0,7: calcolato 0.9526, nel libro 0.95


## Senza piega, uno strato solo

*Nel racconto: «senza scogliera i due neuroni si annullano»; «cento strati valgono uno».*

In [10]:
w_unico = w2 @ W1
b_unico = float(w2 @ b1) + b2
for nome, v, atteso in zip(["energia", "pioggia", "compagno"], w_unico, [9.0, -4.0, 1.5]):
    controlla(f"neurone unico, peso {nome}", v, atteso)
controlla("neurone unico, bias", b_unico, -3.0)
controlla("prima serata", float(w_unico @ [0.8, 1, 1]) + b_unico, 1.7)
controlla("seconda serata, senza piega", float(w_unico @ [0.8, 0, 1]) + b_unico, 5.7)
controlla("seconda serata, con la piega", rete((0.8, 0, 1))[1], 1.9)

✓  neurone unico, peso energia: calcolato 9, nel libro 9
✓  neurone unico, peso pioggia: calcolato -4, nel libro -4
✓  neurone unico, peso compagno: calcolato 1.5, nel libro 1.5
✓  neurone unico, bias: calcolato -3, nel libro -3
✓  prima serata: calcolato 1.7, nel libro 1.7
✓  seconda serata, senza piega: calcolato 5.7, nel libro 5.7
✓  seconda serata, con la piega: calcolato 1.9, nel libro 1.9


## Profondità e approssimazione universale

*Nel racconto: dalle onde il vento, dal vento il tempo; «una rete può imparare qualsiasi cosa?».*

In [11]:
tenda = lambda x: 2 * np.maximum(0, x) - 4 * np.maximum(0, x - 0.5)
for x, atteso in [(0, 0), (0.5, 1), (1, 0)]:
    controlla(f"t({x})", tenda(x), atteso)
controlla("t(t(0,25))", tenda(tenda(0.25)), 1)
controlla("t(t(0,5))", tenda(tenda(0.5)), 0)

def pezzi(k, punti=2 ** 14 + 1):
    x = np.linspace(0, 1, punti)
    for _ in range(k):
        x = tenda(x)
    pendenza = np.sign(np.diff(x))
    return 1 + int(np.sum(pendenza[1:] != pendenza[:-1]))
for k in [1, 2, 3, 10]:
    conta(f"la tenda applicata {k} volte: pezzi dritti", pezzi(k), 2 ** k)
conta("k = 10: neuroni in profondità", 2 * 10, 20)
conta("k = 10: neuroni in larghezza, almeno", 2 ** 10 - 1, 1023)

✓  t(0): calcolato 0, nel libro 0
✓  t(0.5): calcolato 1, nel libro 1
✓  t(1): calcolato 0, nel libro 0
✓  t(t(0,25)): calcolato 1, nel libro 1
✓  t(t(0,5)): calcolato 0, nel libro 0
✓  la tenda applicata 1 volte: pezzi dritti: calcolato 2, nel libro 2
✓  la tenda applicata 2 volte: pezzi dritti: calcolato 4, nel libro 4
✓  la tenda applicata 3 volte: pezzi dritti: calcolato 8, nel libro 8
✓  la tenda applicata 10 volte: pezzi dritti: calcolato 1.024, nel libro 1.024
✓  k = 10: neuroni in profondità: calcolato 20, nel libro 20
✓  k = 10: neuroni in larghezza, almeno: calcolato 1.023, nel libro 1.023


## Le altre attivazioni

*Nel racconto: «scogliere con lo spigolo smussato, vele un po' diverse».*

In [12]:
Phi = lambda z: 0.5 * (1 + math.erf(z / math.sqrt(2)))
z = -0.3
controlla("Φ(−0,3)", Phi(z), 0.382, 0.001)
controlla("GELU(−0,3)", z * Phi(z), -0.115, 0.001)
controlla("SiLU(−0,3)", z * sigmoide(z), -0.128, 0.001)
controlla("σ(−0,6)", sigmoide(-0.6), 0.354, 0.001)
controlla("tanh(−0,3) = 2σ(−0,6) − 1", 2 * sigmoide(-0.6) - 1, -0.291, 0.001)
controlla("tanh(−0,3), calcolata direttamente", math.tanh(z), -0.291, 0.001)

✓  Φ(−0,3): calcolato 0.3821, nel libro 0.382
✓  GELU(−0,3): calcolato -0.1146, nel libro -0.115
✓  SiLU(−0,3): calcolato -0.1277, nel libro -0.128
✓  σ(−0,6): calcolato 0.3543, nel libro 0.354
✓  tanh(−0,3) = 2σ(−0,6) − 1: calcolato -0.2913, nel libro -0.291
✓  tanh(−0,3), calcolata direttamente: calcolato -0.2913, nel libro -0.291


## La temperatura

*Nel racconto: la scotta; dojo, parco e divano a temperatura 0,5, 1 e 2.*

In [13]:
logit = np.array([2.0, 1.0, 0.1])
for T, esp_attesi, somma, prob in [(0.5, [54.60, 7.39, 1.22], 63.21, [0.864, 0.117, 0.019]),
                                   (2,   [2.72, 1.65, 1.05],  5.42,  [0.50, 0.30, 0.19])]:
    e = np.exp(logit / T)
    for v, a in zip(e, esp_attesi):
        controlla(f"T = {T}: esponenziale", v, a)
    controlla(f"T = {T}: somma", e.sum(), somma)
    for nome, v, a in zip(["dojo", "parco", "divano"], softmax(logit / T), prob):
        controlla(f"T = {T}: {nome}", v, a, 0.005)

# la tabella del racconto, in percentuali
for T, attesi in [(0.5, [86, 12, 2]), (1, [66, 24, 10]), (2, [50, 30, 19])]:
    for nome, v, a in zip(["dojo", "parco", "divano"], softmax(logit / T) * 100, attesi):
        controlla(f"tabella, T = {T}: {nome} (%)", v, a, 0.5)

controlla("dojo / parco con T = 0,5", math.exp(1 / 0.5), 7.4, 0.05)
controlla("dojo / parco con T = 2", math.exp(1 / 2), 1.65)

# top-p = 0,9 con T = 1: dojo più parco fanno 0,90, il divano è scartato
p = softmax(logit)
controlla("dojo + parco", p[0] + p[1], 0.90)
controlla("dojo dopo top-p (%)", p[0] / (p[0] + p[1]) * 100, 73, 0.5)
controlla("parco dopo top-p (%)", p[1] / (p[0] + p[1]) * 100, 27, 0.5)

✓  T = 0.5: esponenziale: calcolato 54.6, nel libro 54.6
✓  T = 0.5: esponenziale: calcolato 7.389, nel libro 7.39
✓  T = 0.5: esponenziale: calcolato 1.221, nel libro 1.22
✓  T = 0.5: somma: calcolato 63.21, nel libro 63.21
✓  T = 0.5: dojo: calcolato 0.8638, nel libro 0.864
✓  T = 0.5: parco: calcolato 0.1169, nel libro 0.117
✓  T = 0.5: divano: calcolato 0.01932, nel libro 0.019
✓  T = 2: esponenziale: calcolato 2.718, nel libro 2.72
✓  T = 2: esponenziale: calcolato 1.649, nel libro 1.65
✓  T = 2: esponenziale: calcolato 1.051, nel libro 1.05
✓  T = 2: somma: calcolato 5.418, nel libro 5.42
✓  T = 2: dojo: calcolato 0.5017, nel libro 0.5
✓  T = 2: parco: calcolato 0.3043, nel libro 0.3
✓  T = 2: divano: calcolato 0.194, nel libro 0.19
✓  tabella, T = 0.5: dojo (%): calcolato 86.38, nel libro 86
✓  tabella, T = 0.5: parco (%): calcolato 11.69, nel libro 12
✓  tabella, T = 0.5: divano (%): calcolato 1.932, nel libro 2
✓  tabella, T = 1: dojo (%): calcolato 65.9, nel libro 66
✓  tabel

## La soglia che costa meno

*Nel racconto: il doppio ormeggio, mezz'ora contro cento ore.*

In [14]:
C_fa, C_m = 0.5, 100          # ore: falso allarme, caso mancato
controlla("la soglia p*", C_fa / (C_fa + C_m), 0.005, 0.0005)
p = 0.10
controlla("stasera, non raddoppiare costa in media (ore)", p * C_m, 10)
controlla("stasera, raddoppiare costa in media (ore)", (1 - p) * C_fa, 0.45)

✓  la soglia p*: calcolato 0.004975, nel libro 0.005
✓  stasera, non raddoppiare costa in media (ore): calcolato 10, nel libro 10
✓  stasera, raddoppiare costa in media (ore): calcolato 0.45, nel libro 0.45


## Precision e recall

*Nel racconto: raddoppiare a ogni brezza, o solo quando sei quasi sicuro.*

Cento notti, con l'ancora che cede in 5.

In [15]:
for soglia, VP, FP, FN, prec, rec, costo in [("0,5%", 5, 35, 0, 12.5, 100, 17.5), ("50%", 2, 1, 3, 67, 40, 300.5)]:
    conta(f"soglia {soglia}: notti di allarme", VP + FP, 40 if soglia == "0,5%" else 3)
    controlla(f"soglia {soglia}: precision (%)", VP / (VP + FP) * 100, prec, 0.5)
    controlla(f"soglia {soglia}: recall (%)", VP / (VP + FN) * 100, rec)
    controlla(f"soglia {soglia}: costo (ore)", FP * C_fa + FN * C_m, costo)
print(f"   la soglia bassa costa {300.5 / 17.5:.1f} volte meno: «circa diciassette»")

✓  soglia 0,5%: notti di allarme: calcolato 40, nel libro 40
✓  soglia 0,5%: precision (%): calcolato 12.5, nel libro 12.5
✓  soglia 0,5%: recall (%): calcolato 100, nel libro 100
✓  soglia 0,5%: costo (ore): calcolato 17.5, nel libro 17.5
✓  soglia 50%: notti di allarme: calcolato 3, nel libro 3
✓  soglia 50%: precision (%): calcolato 66.67, nel libro 67
✓  soglia 50%: recall (%): calcolato 40, nel libro 40
✓  soglia 50%: costo (ore): calcolato 300.5, nel libro 300.5
   la soglia bassa costa 17.2 volte meno: «circa diciassette»


## La calibrazione

*Nel racconto: il bollettino del mare; «un 85% che vale 60%».*

In [16]:
fasce = [(100, 0.85, 60 / 100), (100, 0.30, 30 / 100)]      # casi, confidenza, frequenza reale
N = sum(n for n, _, _ in fasce)
controlla("ECE", sum(n / N * abs(freq - conf) for n, conf, freq in fasce), 0.125, 0.0005)
controlla("σ(1,7)", sigmoide(1.7), 0.85)
controlla("con T = 2: σ(0,85)", sigmoide(1.7 / 2), 0.70)
controlla("ln(0,6 / 0,4)", math.log(0.6 / 0.4), 0.405, 0.001)
T = 1.7 / math.log(0.6 / 0.4)
controlla("T che porta 0,85 a 0,60", T, 4.2, 0.05)
controlla("   verifica: σ(1,7 / T)", sigmoide(1.7 / T), 0.60)

✓  ECE: calcolato 0.125, nel libro 0.125
✓  σ(1,7): calcolato 0.8455, nel libro 0.85
✓  con T = 2: σ(0,85): calcolato 0.7006, nel libro 0.7
✓  ln(0,6 / 0,4): calcolato 0.4055, nel libro 0.405
✓  T che porta 0,85 a 0,60: calcolato 4.193, nel libro 4.2
✓     verifica: σ(1,7 / T): calcolato 0.6, nel libro 0.6


## Alberi, foreste e l'altro albero

*Nel racconto: la katana forgiata sul diario.*

In [17]:
gini = lambda si, no: 1 - (si / (si + no)) ** 2 - (no / (si + no)) ** 2
controlla("dieci serate, cinque andate: impurità", gini(5, 5), 0.5)
controlla("energia sopra 0,5: 6 serate, 5 andate", gini(5, 1), 0.278, 0.001)
controlla("energia sotto: 4 serate, nessuna andata", gini(0, 4), 0)
controlla("impurità dopo la domanda", 6 / 10 * gini(5, 1) + 4 / 10 * gini(0, 4), 0.167, 0.001)

# l'altro albero: la regola UCB della Monte Carlo Tree Search
c, N = 1.41, 12
ucb = lambda v, n: (v / n, c * math.sqrt(math.log(N) / n))
controlla("ln 12", math.log(N), 2.485, 0.001)
for nome, v, n, medio, premio, totale in [("A", 6, 10, 0.6, 0.70, 1.30), ("B", 1, 2, 0.5, 1.57, 2.07)]:
    m, e = ucb(v, n)
    controlla(f"ramo {nome}: vittorie su prove", m, medio)
    controlla(f"ramo {nome}: premio per l'esplorazione", e, premio)
    controlla(f"ramo {nome}: punteggio", m + e, totale)

✓  dieci serate, cinque andate: impurità: calcolato 0.5, nel libro 0.5
✓  energia sopra 0,5: 6 serate, 5 andate: calcolato 0.2778, nel libro 0.278
✓  energia sotto: 4 serate, nessuna andata: calcolato 0, nel libro 0
✓  impurità dopo la domanda: calcolato 0.1667, nel libro 0.167
✓  ln 12: calcolato 2.485, nel libro 2.485
✓  ramo A: vittorie su prove: calcolato 0.6, nel libro 0.6
✓  ramo A: premio per l'esplorazione: calcolato 0.7029, nel libro 0.7
✓  ramo A: punteggio: calcolato 1.303, nel libro 1.3
✓  ramo B: vittorie su prove: calcolato 0.5, nel libro 0.5
✓  ramo B: premio per l'esplorazione: calcolato 1.572, nel libro 1.57
✓  ramo B: punteggio: calcolato 2.072, nel libro 2.07


## Il modello decisionale

*Nel racconto: le bandiere di segnalazione.*

In [18]:
bandiere = ["chiede un ormeggio", "segnala un pericolo", "altro"]
s = np.array([2.0, 0.5, 0.0])
for v, a in zip(np.exp(s), [7.39, 1.65, 1]):
    controlla("esponenziale", v, a)
controlla("somma", np.exp(s).sum(), 10.04)
for nome, v, a in zip(bandiere, softmax(s) * 100, [73.6, 16.4, 10.0]):
    controlla(f"«{nome}» (%)", v, a, 0.05)

✓  esponenziale: calcolato 7.389, nel libro 7.39
✓  esponenziale: calcolato 1.649, nel libro 1.65
✓  esponenziale: calcolato 1, nel libro 1
✓  somma: calcolato 10.04, nel libro 10.04
✓  «chiede un ormeggio» (%): calcolato 73.61, nel libro 73.6
✓  «segnala un pericolo» (%): calcolato 16.43, nel libro 16.4
✓  «altro» (%): calcolato 9.962, nel libro 10


## Kata per casa

**Esercizio 1.** La softmax dei logit (3; 1; 0), prima così com'è e poi con temperatura 2.

In [19]:
logit = np.array([3.0, 1.0, 0.0])
for v, a in zip(np.exp(logit), [20.09, 2.72, 1.00]):
    controlla("e^logit", v, a)
controlla("somma", np.exp(logit).sum(), 23.80)
for i, (v, a) in enumerate(zip(softmax(logit) * 100, [84, 11, 4]), 1):
    controlla(f"opzione {i} (%)", v, a, 0.5)
for v, a in zip(np.exp(logit / 2), [4.48, 1.65, 1.00]):
    controlla("e^(logit/2)", v, a)
controlla("somma, T = 2", np.exp(logit / 2).sum(), 7.13)
for i, (v, a) in enumerate(zip(softmax(logit / 2) * 100, [63, 23, 14]), 1):
    controlla(f"opzione {i}, T = 2 (%)", v, a, 0.5)

✓  e^logit: calcolato 20.09, nel libro 20.09
✓  e^logit: calcolato 2.718, nel libro 2.72
✓  e^logit: calcolato 1, nel libro 1
✓  somma: calcolato 23.8, nel libro 23.8
✓  opzione 1 (%): calcolato 84.38, nel libro 84
✓  opzione 2 (%): calcolato 11.42, nel libro 11
✓  opzione 3 (%): calcolato 4.201, nel libro 4
✓  e^(logit/2): calcolato 4.482, nel libro 4.48
✓  e^(logit/2): calcolato 1.649, nel libro 1.65
✓  e^(logit/2): calcolato 1, nel libro 1
✓  somma, T = 2: calcolato 7.13, nel libro 7.13
✓  opzione 1, T = 2 (%): calcolato 62.85, nel libro 63
✓  opzione 2, T = 2 (%): calcolato 23.12, nel libro 23
✓  opzione 3, T = 2 (%): calcolato 14.02, nel libro 14


**Esercizio 2.** La serata (0,3; 1; 0): stanco, sotto la pioggia, senza compagno.

In [20]:
h, z, y = rete((0.3, 1, 0))
controlla("lo slancio (spento)", h[0], 0)
controlla("la pioggia che pesa", h[1], 1.6)
controlla("spinta", z, -3.7)
controlla("probabilità", y, 0.02)

✓  lo slancio (spento): calcolato 0, nel libro 0
✓  la pioggia che pesa: calcolato 1.6, nel libro 1.6
✓  spinta: calcolato -3.7, nel libro -3.7
✓  probabilità: calcolato 0.02413, nel libro 0.02


## Prova tu

Tira e molla la scotta: che cosa succede al favorito quando T va verso zero? E quando cresce molto?

In [21]:
for T in [0.25, 0.5, 1, 2, 5]:
    print(f"T = {T}:", ", ".join(f"{v:.0%}" for v in softmax(logit / T)))

T = 0.25: 100%, 0%, 0%
T = 0.5: 98%, 2%, 0%
T = 1: 84%, 11%, 4%
T = 2: 63%, 23%, 14%
T = 5: 45%, 30%, 25%
